# Lab 8: Law, Ethics, and the Closing Web

**DSA 405 · Week 8**

| | |
|---|---|
| **In class** | Friday, Oct 9 |
| **A8 due** | Thursday, Oct 15, 11:59 PM |
| **Reference** | `DSA405_Wk08_CaseSet_FA26.md` (we go through it in class; keep it open for Task 1.2) |
| **Time** | ~25 min in class, ~60 min at home |

## Overview

Week 9 introduces the scraping tools. Scraping means writing a program that downloads
web pages and collects data from them. This week we learn the rules that say which
sites and data sources those tools may be used on. We do it in this order for a
practical reason. It is easy to build a project on a data source that turns out to be
one we are not allowed to use, and rewriting a data-collection plan in Week 11 takes
time you will not have. Reading the rules first prevents this.

A8 also prepares you for P3: the **acquisition plan** you write here (your written
plan for how you will collect each data source) is the plan you carry out in Weeks 9
through 11.

This material is education, not legal advice. The law in this area is unsettled and
still changing.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, time, io
from bs4 import BeautifulSoup

# Tell the site who you are. Many sites refuse requests from scripts that do not identify themselves.
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}

pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| bs4 ready")

---
# Part 1: Explore (in class)

## Task 1.1: Read a `robots.txt`

A website's rules for automated programs are published in a plain-text file at
`/robots.txt`. Reading a few of these files is the fastest way to learn the vocabulary
they use. Fetch one first:

In [ ]:
def get_robots(domain):
    """Fetch and print a site's robots.txt."""
    url = f"https://{domain}/robots.txt"
    r = requests.get(url, headers=HEADERS, timeout=20)
    print(f"{url}  ->  HTTP {r.status_code}\n")
    print(r.text[:2500] if r.ok else "(no robots.txt served)")
    return r.text if r.ok else ""


robots = get_robots("en.wikipedia.org")

Now fetch two or three more. These sites have `robots.txt` files that are good
examples to read: `www.reddit.com`, `www.nytimes.com`, `github.com`,
`books.toscrape.com`, and the site you are using for your project.

For each one, record what is disallowed, whether there is a `Crawl-delay`, and
anything that surprises you. One of these sites answers with HTTP 404, which means
it publishes no `robots.txt` at all. Write down which one; we use it in the next
step.

In [ ]:
# fetch two or three more

Python's standard library includes a `robots.txt` parser, so you do not have to read
and interpret the files yourself:

In [ ]:
from urllib.robotparser import RobotFileParser

rp = RobotFileParser()
rp.set_url("https://books.toscrape.com/robots.txt")
rp.read()

for path in ["/", "/catalogue/page-1.html", "/admin/"]:
    print(f"{path:32} allowed: {rp.can_fetch(HEADERS['User-Agent'], path)}")

Every path came back `True`, including `/admin/`. That is not because the site
owner allowed it. `books.toscrape.com` serves no `robots.txt` (HTTP 404), and when the
file is missing, the parser answers `True` for every path. `True` here means "no rule
forbids this," not "the owner said yes." A missing file is not permission. The site's
terms of service and the six standing rules from the reading still apply.

Two more things to notice about `can_fetch`. First, it only answers for the user-agent
you pass it, so pass the name your code really sends. Second, it ignores `Crawl-delay`;
you have to read that line yourself and add the pause to your code.

Here is an example how to read the `crawl-delay` for a site:

In [ ]:
rp.set_url("https://www.nytimes.com/robots.txt")
rp.read()
delay = rp.crawl_delay(HEADERS['User-Agent'])

if delay is not None:
    print(f"Crawl-delay: {delay} seconds")
else:
    print("No Crawl-delay specified for this user-agent.")

## Task 1.2: Apply the four buckets

The case set describes four bodies of law that apply when you collect data from the
web. We call these four categories **buckets**. To analyze a scenario, you decide
which buckets apply to it.

Buckets: **1 Hacking (CFAA)** · **2 Contract (ToS)** · **3 Circumvention (DMCA §1201)** ·
**4 Privacy (GDPR/CCPA)**

Read the **six scenarios** from the case set as a team. For each scenario, fill in one
row of the table.

| # | Buckets that apply | Proceed? | What would have to change |
|---|---|---|---|
| A | | | |
| B | | | |
| C | | | |
| D | | | |
| E | | | |
| F | | | |

**As a team, rank the six scenarios from safest to most dangerous** and be ready to
explain your ordering. One scenario is riskier than it first appears, and most teams
put it in the wrong place on their first ranking, so take time to discuss the order
rather than deciding it quickly.

## Task 1.3: Check the project source

Fetch the `robots.txt` for the site or API you plan to use in P3 (or a site you already used for P2), and run the parser
against the exact paths you intend to fetch. (An **API** is a service that sends data
directly to a program that requests it, instead of showing a web page. Week 11 covers
APIs in full.)

If your source is a file download or an API rather than a page you scrape, find its
**license** instead and paste the relevant sentence.

In [ ]:
# your project's source

---
## Checkpoint: submit before leaving

1. Which site's `robots.txt` surprised you the most, and why?
2. Your team's ranking of the six scenarios, from safest to most dangerous.
3. Which scenario did the team discuss the longest?
4. One sentence: is your P3 source clearly allowed, unclear, or a problem?

*Answers here.*

---
# Part 2: A8, Acquisition Plan and Ethics Review

This document becomes your P3 plan, so write it as a plan you will actually follow.

## Task 2.1: Acquisition plan

For **each** source in your project, complete the table.

### Source 1

| | |
|---|---|
| **Publisher** | |
| **URL** | |
| **What it contains** | |
| **Access method** | download / `read_html` / `requests` + BeautifulSoup / REST API |
| **Approximate volume** | how many pages or requests, total |
| **Request rate** | |
| **Does it contain personal data?** | |
| **Requires a login?** | must be **no** |

**Applicable constraints, quoted:**

- `robots.txt`: paste the lines that apply to your plan, or state that none apply
- Terms of service: quote the sentence that governs automated access, and give a link
- License: name the license, and state what attribution it requires

### Source 2

*(same table)*

## Task 2.2: Bucket analysis

For the **whole project**, go through the four buckets and fill in the table:

| Bucket | Applies? | Reasoning |
|---|---|---|
| 1. Hacking (CFAA) | | Does the plan access anything that requires permission, such as a login? |
| 2. Contract (ToS) | | Were the terms accepted by clicking "I agree" (clickwrap), or are they only posted on the site? |
| 3. Circumvention (§1201) | | Does the plan bypass any protection, such as a rate limit (a cap on how many requests a site accepts from you) or bot detection? |
| 4. Privacy | | Is any of the data about identifiable individuals? |

A "yes" on bucket 1, 3, or 4 means **the plan needs to change**. 

## Task 2.3: Guardrail certification

A guardrail is a rule that keeps your data collection safe and allowed. Check every
box. If you cannot check one, explain why.

- [ ] My project source is a sandbox (a practice site built for scraping), a documented API, an
  open-licensed source, or a site whose robots.txt and terms permit it
- [ ] My project data is not behind a login or paywall (a page that requires payment to view)
- [ ] I am not collecting personal or identifiable information about individuals
- [ ] Requests are rate-limited (your code pauses between requests), the user-agent
  (the name your code sends with each request) identifies you honestly, and responses
  are cached (saved on your computer) so the same resource is never fetched twice
- [ ] You asked about any open questions before submitting

## Task 2.4: Prove the plan works

Many of you have already done the steps below for P1. Feel free to copy code from your P1 submission.

Write code that:

1. Fetches **one** page or makes **one** API call from each source
2. Prints the HTTP status
3. Shows enough of the response to confirm the target data is present
4. Uses the honest user-agent
5. Sleeps (pauses) between requests

One page, not the whole site. This is a feasibility test, which means a small check
that the plan can work at all. 

In [ ]:
# make one request per source, and pause between requests

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A8_[yourUnityID].ipynb`
4. Upload to the **A8** space on Moodle.

The **Checkpoint** section goes separately to **Week 8 In-Class Activity** before
leaving class on Friday. A8 is due **Thursday, Oct 15, 11:59 PM**.